Processing Orders Data using python

In [0]:
import dlt
from pyspark.sql import functions as F

In [0]:
@dlt.table(
    name = "bronze_orders",
    comment = "bronze raw layer for orders",
    table_properties = {"quality": "bronze"}
)

def create_bronze_order():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","json")
        .option("cloudFiles.inferColumnTypes","true")
        .load("/Volumes/circuitbox/landing/operational_data/orders/")
        .select(
            "*",
            F.col("_metadata.file_path").alias("file_path"),
            F.current_timestamp().alias("ingest_timestamp")
        )
    )

In [0]:
@dlt.table(
    name = "silver_orders_clean",
    comment = "silver clean layer",
    table_properties = {"quality": "silver"}
)

@dlt.expect_or_fail("valid_customer_id","customer_id is not null")
@dlt.expect_or_fail("valid_order_id","order_id is not null")
@dlt.expect("valid_order_status","order_status IN ('Pending','Shipped','Cancelled','Completed')")
@dlt.expect("valid_payment_method","payment_method IN ('Credit Card','PayPal','Bank Transfer')")

def create_silver_orders_clean():
    return(
        spark.readStream.table("LIVE.bronze_orders")
        .select(
            "customer_id",
            "items",
            "order_id",
            "order_status",
            F.col("order_timestamp").cast("timestamp"),
            "payment_method"
        )       
    )


In [0]:
dlt.create_streaming_table(
    name = "silver_orders",
    comment = "silver clean layer SCDP typ 2",
    table_properties = {"quality": "silver"}
)

dlt.apply_changes(
    target = "silver_orders",
    source = "silver_orders_clean",
    keys = ["order_id"],
    sequence_by = "order_timestamp", 
    stored_as_scd_type = 2
    )
